# Day 6 · Exercise 4: Format-Constrained Prompt

**What you'll build:** `add_format_constraint(base_prompt: str, format_type: str, max_items: int | None) -> str` — a function that appends precise format instructions (and the right sentinel prefix) to any base prompt so the model's output is predictable enough to parse downstream.

**Why it matters:** Controlling output shape is the last mile of prompt engineering — you can have perfect content but still break a pipeline if the model returns prose when you needed JSON. Mastering format constraints means every function you write from here on can reliably hand structured data to the next stage.

## Your Implementation

In [ ]:
def add_format_constraint(
    base_prompt: str,
    format_type: str,
    max_items: int | None = None,
) -> str:
    """Append format instructions and a sentinel prefix to a base prompt.

    Supports four format types:
      - "bullet"   : hyphen-prefixed bullet list
      - "numbered" : numbered list starting with "1."
      - "json"     : raw JSON object (no markdown fences), sentinel "{"
      - "sentence" : exactly one sentence

    When `max_items` is provided (and the format produces a list), the
    instruction specifies the exact item count.  For "sentence" and "json"
    `max_items` is ignored.

    The returned string is the full prompt ready to be sent to the model.
    It ends with a sentinel token that locks in the first character of the
    model's response so it cannot drift into prose preamble.

    Args:
        base_prompt: The core task or question for the model.
        format_type: One of "bullet", "numbered", "json", "sentence".
        max_items:   Number of list items to request (lists only); ignored
                     for "json" and "sentence".

    Returns:
        The base_prompt with format instructions and a sentinel appended.

    Example:
        >>> add_format_constraint("List benefits of water.", "bullet", 3)
        'List benefits of water.\n\nReturn a bullet list. Each item starts with \'- \'. List exactly 3 items. Do not add any heading, intro, or conclusion.\n\n- '
    """
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────

## Check Your Work

Run the cell below — it runs 5 automated checks and shows ✅ / ❌ for each.

In [ ]:
_PASS, _FAIL = '✅', '❌'

def _run_checks():
    score, total = 0, 5

    # Check 1: function exists and is callable
    try:
        assert callable(add_format_constraint), 'add_format_constraint is not defined'
        result_type = type(add_format_constraint('hello', 'sentence'))
        assert result_type is str, f'expected str return type, got {result_type.__name__}'
        print(f'{_PASS} Check 1/{total}: function exists, is callable, and returns a str')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 1/{total}: {e}')
        return

    # Check 2: bullet format — contains format instruction and sentinel "- "
    try:
        prompt = add_format_constraint('List benefits of water.', 'bullet', 3)
        assert isinstance(prompt, str), 'return value must be a str'
        assert '- ' in prompt, 'bullet format must include "- " in the output'
        assert prompt.endswith('- '), (
            f'bullet prompt must end with sentinel "- "; got: {prompt[-20:]!r}'
        )
        assert '3' in prompt, 'bullet format with max_items=3 must mention 3 in the instructions'
        print(f'{_PASS} Check 2/{total}: bullet format includes instructions, max_items count, and ends with sentinel "- "')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 2/{total}: {e}')

    # Check 3: numbered format — ends with sentinel "1."
    try:
        prompt = add_format_constraint('List tips for sleep.', 'numbered', 4)
        assert isinstance(prompt, str), 'return value must be a str'
        stripped = prompt.rstrip()
        assert stripped.endswith('1.'), (
            f'numbered prompt must end with sentinel "1."; got: {prompt[-20:]!r}'
        )
        assert '4' in prompt, 'numbered format with max_items=4 must mention 4 in the instructions'
        print(f'{_PASS} Check 3/{total}: numbered format includes instructions, max_items count, and ends with sentinel "1."')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 3/{total}: {e}')

    # Check 4: json format — ends with sentinel "{"
    try:
        prompt = add_format_constraint('Extract name and age from the text.', 'json')
        assert isinstance(prompt, str), 'return value must be a str'
        stripped = prompt.rstrip()
        assert stripped.endswith('{'), (
            f'json prompt must end with sentinel "{{" ; got: {prompt[-20:]!r}'
        )
        print(f'{_PASS} Check 4/{total}: json format ends with sentinel "{{", no max_items needed')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 4/{total}: {e}')

    # Check 5: sentence format — base_prompt is preserved and instructions added
    try:
        base = 'What is the main benefit of exercise?'
        prompt = add_format_constraint(base, 'sentence')
        assert isinstance(prompt, str), 'return value must be a str'
        assert base in prompt, 'base_prompt must be preserved in the returned string'
        lower = prompt.lower()
        assert 'sentence' in lower or 'one sentence' in lower, (
            'sentence format must instruct the model to reply in one sentence'
        )
        print(f'{_PASS} Check 5/{total}: sentence format preserves base_prompt and includes sentence instruction')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 5/{total}: {e}')

    print()
    if score == total:
        print('🎉 Exercise complete!')
    print(f'  {score}/{total} passed.' + ('' if score == total else ' Keep going!'))

_run_checks()

## Bonus Challenge

In Day 8 you will write retry logic that calls the model, checks whether the output parses correctly, and re-calls if it does not. Try wiring that up now:

```python
import ollama, json

def extract_json_with_retry(text: str, max_attempts: int = 3) -> dict:
    """Build a JSON-constrained prompt, call the model, parse, retry on failure."""
    base = f"Extract the name and city from this text: {text}"
    prompt = add_format_constraint(base, 'json')
    for attempt in range(1, max_attempts + 1):
        response = ollama.chat(
            model='llama3.2',
            messages=[{'role': 'user', 'content': prompt}],
        )
        raw = response['message']['content'].strip()
        # Ensure the sentinel is present
        if not raw.startswith('{'):
            raw = '{' + raw
        # Find the closing brace and trim trailing noise
        end = raw.rfind('}')
        if end != -1:
            raw = raw[:end + 1]
        try:
            return json.loads(raw)
        except json.JSONDecodeError:
            print(f'Attempt {attempt} failed to parse; retrying...')
    raise ValueError('Model did not return valid JSON after all retries.')
```

Run it on a sample sentence and verify `isinstance(result, dict)` is `True`.

## Solution

<details>
<summary>Click to reveal — try on your own first</summary>

```python
def add_format_constraint(
    base_prompt: str,
    format_type: str,
    max_items: int | None = None,
) -> str:
    """Append format instructions and a sentinel prefix to a base prompt."""
    if format_type == 'bullet':
        count_clause = f'exactly {max_items} items' if max_items is not None else 'the items'
        instruction = (
            f"Return a bullet list. Each item starts with '- '. "
            f"List {count_clause}. Do not add any heading, intro, or conclusion."
        )
        sentinel = '- '
    elif format_type == 'numbered':
        count_clause = f'exactly {max_items} items' if max_items is not None else 'the items'
        instruction = (
            f"Return a numbered list. Each item starts with its number, a period, "
            f"and a space. List {count_clause}. No heading, no intro, no conclusion."
        )
        sentinel = '1.'
    elif format_type == 'json':
        instruction = (
            "Return a raw JSON object — no markdown code fences, no explanation, "
            "no text before or after the JSON object."
        )
        sentinel = '{'
    elif format_type == 'sentence':
        instruction = (
            "Reply in exactly one sentence. "
            "Do not use bullet points, lists, or multiple sentences."
        )
        sentinel = ''
    else:
        raise ValueError(f"Unknown format_type: {format_type!r}")

    parts = [base_prompt, instruction]
    if sentinel:
        parts.append(sentinel)
        return '\n\n'.join(parts[:-1]) + '\n\n' + sentinel
    return '\n\n'.join(parts)
```

**Why this works:** Each branch builds an instruction that names the format, specifies every structural detail (item prefix, exact count, no preamble), and ends with a sentinel token that forces the model to open in the correct format — a left brace for JSON, `- ` for bullets, `1.` for numbered lists. Because the model generates tokens left-to-right and the sentinel is the last thing it sees, it has no room to add a preamble before the format begins. The `sentence` case omits the sentinel because prose has no unambiguous opening character; precise instructions alone are sufficient there.
</details>